# Airbnb First Booking — Pipeline Walkthrough

> Multiclass first-booking destination ranking (NDCG@5).

This notebook walks through the production pipeline using the modules in `src/`. The model is loaded from the serialized artifact produced by `python -m src.pipeline`.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_columns", 50)

## 1. Data

Load the versioned sample and inspect it.

In [2]:
df = pd.read_csv(config.SAMPLE_PATH)
print(df.shape, "| classes:", df[config.TARGET].nunique())
df.head()

(20001, 16) | classes: 12


,id,date_account_created,timestamp_first_active,date_first_booking,gender,age,signup_method,signup_flow,language,affiliate_channel,affiliate_provider,first_affiliate_tracked,signup_app,first_device_type,first_browser,country_destination
0,i4ibg6hdkb,2014-05-08,20140508010903,2014-05-08,MALE,NaN,basic,0,en,sem-non-brand,google,omg,Web,Mac Desktop,Safari,AU
1,x22igsy5xl,2012-06-23,20120623235637,2012-06-24,FEMALE,NaN,basic,0,en,sem-non-brand,vast,omg,Web,Windows Desktop,IE,AU
2,5m97yjzhl1,2013-11-21,20131121041307,2013-12-07,MALE,24.0,facebook,0,en,direct,direct,linked,Web,Windows Desktop,Firefox,AU
3,t4wk57eon2,2012-08-06,20120806175524,2012-08-10,MALE,27.0,basic,0,en,direct,direct,linked,Web,Mac Desktop,Chrome,AU
4,poc1qeu2ao,2014-03-29,20140329031623,2014-03-30,FEMALE,24.0,basic,0,en,direct,direct,untracked,Web,Mac Desktop,Safari,AU


## 2. Preprocessing

The same transform used in training and serving.

In [3]:
from src.preprocessing import Preprocessor
X, y = Preprocessor().run(df)
print("features:", X.shape, "| destinations:", y.nunique())
X.head()

features: (20001, 13) | destinations: 12


,date_account_created,timestamp_first_active,gender,age,signup_method,signup_flow,language,affiliate_channel,affiliate_provider,first_affiliate_tracked,signup_app,first_device_type,first_browser
0,2014-05-08,20140508010903,MALE,NaN,basic,0,en,sem-non-brand,google,omg,Web,Mac Desktop,Safari
1,2012-06-23,20120623235637,FEMALE,NaN,basic,0,en,sem-non-brand,vast,omg,Web,Windows Desktop,IE
2,2013-11-21,20131121041307,MALE,24.0,facebook,0,en,direct,direct,linked,Web,Windows Desktop,Firefox
3,2012-08-06,20120806175524,MALE,27.0,basic,0,en,direct,direct,linked,Web,Mac Desktop,Chrome
4,2014-03-29,20140329031623,FEMALE,24.0,basic,0,en,direct,direct,untracked,Web,Mac Desktop,Safari


## 3. Model and evaluation

Metrics from the serialized model card.

In [4]:
card = json.loads(Path(config.MODEL_CARD_PATH).read_text())
print(json.dumps(card, indent=2)[:1800])

{
  "schema_version": "1.0",
  "trained_at": "2026-06-15T00:03:24+00:00",
  "dataset": "competitions/airbnb-recruiting-new-user-bookings",
  "data_source": "kaggle",
  "data_sha256": "1d57dabe8b06534e51f86520ce6a66b6f67cce1704fb77a3fcfb596124858818",
  "target": "country_destination",
  "problem": "multiclass first-booking destination (12 classes, imbalanced)",
  "best_model": "hist_gb",
  "best_params": {
    "clf__max_leaf_nodes": 15,
    "clf__max_iter": 400,
    "clf__max_depth": 6,
    "clf__learning_rate": 0.03059949687207196
  },
  "cv_selection": [
    {
      "model": "hist_gb",
      "neg_log_loss_mean": -1.0742,
      "neg_log_loss_std": 0.0038
    },
    {
      "model": "logreg",
      "neg_log_loss_mean": -1.1297,
      "neg_log_loss_std": 0.0002
    }
  ],
  "baseline": {
    "model": "always-NDF",
    "accuracy": 0.5835
  },
  "holdout": {
    "accuracy": 0.6303,
    "top5_accuracy": 0.9588,
    "ndcg5": 0.8236,
    "macro_f1": 0.1023,
    "n_classes": 12
  },
  "busine

## 4. Prediction

The serving contract on a representative input.

In [5]:
from src.predict import Predictor
pred = Predictor()
user = {"gender":"FEMALE","age":34,"signup_method":"facebook","signup_flow":0,"language":"en","affiliate_channel":"direct","affiliate_provider":"direct","first_affiliate_tracked":"untracked","signup_app":"Web","first_device_type":"Mac Desktop","first_browser":"Chrome","date_account_created":"2014-06-01","timestamp_first_active":"20140531120000"}
print("top-5 destinations:", pred.rank(user, k=5))

top-5 destinations: [('NDF', 0.6852331884332515), ('US', 0.20820044895055495), ('other', 0.039966395576383346), ('FR', 0.0193288099116948), ('IT', 0.012533726476582309)]


## Reproduce

Run the full pipeline end to end:

```
python -m src.pipeline
```